# Cairo's Truthful Principle: The Hidden Water Divide

**Companion notebook for the Medium article:**  
[How To Use Cairo's Truthful Principle On The Hidden Water Divide](https://medium.com/@loewenj700)

Two charts built with Python + Plotly from the UN SDG / WHO-UNICEF JMP safe drinking water dataset (2000–2024).

**Dataset:** `SH_H2O_SAFE.csv` — safely managed drinking water services, global and country level, 2000–2024  
**Source:** [WHO/UNICEF Joint Monitoring Programme (JMP)](https://washdata.org/) via UN SDG Global Database  
**Author:** John Loewen | [Medium](https://medium.com/@loewenj700) | [GitHub](https://github.com/loewenj700)

---

## Setup

Install dependencies if needed:

```
pip install pandas plotly
```

In [ ]:
import pandas as pd
import plotly.graph_objects as go

## The Dataset

Load `SH_H2O_SAFE.csv` — 10,780 rows covering 185+ countries and territories, 2000–2024.

Columns we use:

- `area` — country or region name
- `area_code` — UN M49 numeric code
- `year` — year of the estimate
- `value` — % of population with safely managed drinking water
- `location` — `URBAN`, `RURAL`, or `ALLAREA`

"Safely managed" means water from an improved source that is on-premises, available when needed, and free of contamination (JMP definition).

In [ ]:
df = pd.read_csv('SH_H2O_SAFE.csv')
print(df.shape)
print(df.head())

---

## Chart 1: The Gap That Does Not Close

Filter to world-level data and split by urban vs rural access — a 25-year picture of two very different trends.

**Key finding:** World urban access has been essentially flat at ~83.5% since 2000. Rural access climbed from 42% to 60.5% — real progress, but the 23-point gap barely moved.

Presenting only the global average (73.7%) makes this 23-point divide invisible. That is what Cairo means by a chart that is *technically accurate but untruthful*.

In [ ]:
world = df[df['area'] == 'World']
trends = world[world['location'].isin(['URBAN', 'RURAL'])]

urban = trends[trends['location'] == 'URBAN']
rural = trends[trends['location'] == 'RURAL']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=urban['year'], y=urban['value'],
    mode='lines', name='Urban',
    line=dict(color='#0072B2', width=3)
))
fig.add_trace(go.Scatter(
    x=rural['year'], y=rural['value'],
    mode='lines', name='Rural',
    line=dict(color='#D62728', width=3)
))
fig.update_layout(
    title='Urban access stalled; rural access grew — but the gap remains',
    yaxis_ticksuffix='%',
    template='plotly_white'
)
fig.show()

---

## Chart 2: Where The Divide Is Deepest

Filter to countries only (exclude regional aggregates), compute the urban-rural gap for each country in 2024, and show the top 15 worst cases as a lollipop chart.

**Key finding:** Gambia tops the list with a 58-point gap — about 10% of rural Gambians have safely managed water while nearly 70% of urban residents do. A national average for Gambia would look unremarkable. The lollipop makes the inequality impossible to miss.

In [ ]:
AGGREGATE_CODES = {1, 5, 9, 11, 13, 14, 15, 17, 18, 21, 29, 30, 34, 35, 39,
                   53, 54, 57, 61, 62, 142, 143, 145, 150, 151, 154, 155,
                   199, 202, 419, 432, 513, 722, 747, 753}

countries = df[~df['area_code'].isin(AGGREGATE_CODES)]
data_2024 = countries[countries['year'] == 2024]

urban_c = data_2024[data_2024['location'] == 'URBAN'][['area', 'value']].rename(columns={'value': 'urban'})
rural_c = data_2024[data_2024['location'] == 'RURAL'][['area', 'value']].rename(columns={'value': 'rural'})
merged = pd.merge(urban_c, rural_c, on='area').dropna()
merged['gap'] = merged['urban'] - merged['rural']
top15 = merged.sort_values('gap', ascending=False).head(15).sort_values('gap', ascending=True)

fig = go.Figure()
for _, row in top15.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['rural'], row['urban']],
        y=[row['area'], row['area']],
        mode='lines', line=dict(color='#B8B8B8', width=3),
        showlegend=False
    ))
fig.add_trace(go.Scatter(
    x=top15['rural'], y=top15['area'],
    mode='markers', name='Rural',
    marker=dict(color='#E69F00', size=14)
))
fig.add_trace(go.Scatter(
    x=top15['urban'], y=top15['area'],
    mode='markers', name='Urban',
    marker=dict(color='#0072B2', size=14)
))
fig.update_layout(
    title='The biggest urban-rural divides in 2024',
    xaxis_ticksuffix='%', template='plotly_white'
)
fig.show()

---

**Source:** WHO/UNICEF Joint Monitoring Programme for Water Supply, Sanitation and Hygiene (JMP), 2025 update. Via UN SDG Global Database.  
**Article:** [How To Use Cairo's Truthful Principle On The Hidden Water Divide](https://medium.com/@loewenj700)  
**Author:** John Loewen — [Medium](https://medium.com/@loewenj700) | [Substack](https://johnloewen.substack.com/)